# Train from expanded crops (LEGACY)

**Legacy** multi-model path (semantic + image + fusion). Prefer 
otebooks/01-custom-cnn.ipynb for the Grad-CAM-by-CNN workflow.

Pré-requisito: tabela de crops gerada pelo expand:

`	ext
python scripts/expand_lidc_dataset.py run --batch-size 25 --max-gb 8
`

Este notebook chama scripts/train_from_crops.py e grava em models/{semantic_mlp,image_cnn,fusion_cnn_mlp}/.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path("..").resolve()
if not (ROOT / "scripts" / "train_from_crops.py").exists():
    ROOT = Path(".").resolve()

script = ROOT / "scripts" / "train_from_crops.py"
crops_csv = ROOT / "outputs" / "preprocessing" / "lidc_model_table_fixed_crops.csv"

print("ROOT:", ROOT)
print("script:", script)
print("crops_csv exists:", crops_csv.exists())
if crops_csv.exists():
    import pandas as pd
    df = pd.read_csv(crops_csv)
    print(f"nodules={len(df)} patients={df['patient_id'].nunique()}")
    print(df["label"].value_counts())
else:
    raise FileNotFoundError(
        "Rode antes: python scripts/expand_lidc_dataset.py run --batch-size 25 --max-gb 8"
    )

In [ ]:
# Treina os 3 baselines. Ajuste --models se quiser só um, ex: --models fusion
cmd = [
    sys.executable,
    str(script),
    "--models", "semantic,image,fusion",
    "--epochs", "100",
    "--patience", "20",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(ROOT))
if proc.returncode != 0:
    raise RuntimeError(f"train_from_crops.py failed with exit code {proc.returncode}")

In [ ]:
import pandas as pd

rows = []
for path in [
    ROOT / "models" / "semantic_mlp" / "semantic_only_mlp_results.csv",
    ROOT / "models" / "image_cnn" / "image_only_cnn_results.csv",
    ROOT / "models" / "fusion_cnn_mlp" / "fusion_cnn_mlp_test_results.csv",
]:
    if path.exists():
        part = pd.read_csv(path)
        rows.append(part)
        print("\n", path.name)
        display(part)

print("\nWeights:")
for p in [
    ROOT / "models" / "semantic_mlp" / "best_semantic_mlp.pth",
    ROOT / "models" / "image_cnn" / "best_image_cnn.pth",
    ROOT / "models" / "fusion_cnn_mlp" / "best_fusion_cnn_mlp.pth",
]:
    print(" ", p.name, "OK" if p.exists() else "MISSING")